# Notebook 03 — Create Genie Spaces

**What you’ll learn:** How to create and configure a Genie space — the natural-language interface where users ask questions and Genie translates them to SQL.

**This notebook creates three spaces** on the same manufacturing tables:

| Space | What it has | Why |
|-------|------------|-----|
| **Configured** | Full SQL instructions, sample questions, and curated Q-to-SQL examples | Best accuracy — your primary space |
| **No Examples** | SQL instructions included, but no curated examples | For A/B comparison in notebook 07 |
| **Blank** | Tables only, no instructions | Baseline — see how Genie performs with zero guidance |

**Why three?** Notebook 07 runs the same benchmark questions against the "Configured" and "No Examples" spaces to measure how much curated SQL examples improve Genie’s accuracy. The blank space is optional — it shows the difference between no guidance at all vs. business instructions.

**Before you start:** Run notebook **02** (data setup).

**Compute:** Serverless.

## Serverless compute

Attach **Serverless** notebook compute. The notebook uses the Databricks **SDK** and `spark` only for the final `workshop_config` table; config stays in **Python variables** (no `spark.conf.set` for custom keys).

## Configuration

Set **Catalog** and **Schema** to match **notebook 01** (your Unity Catalog location for this workshop). The widget defaults are **examples only**—replace them with your catalog and schema before running in a customer workspace.

All later notebooks use the **same** widget names so the path stays consistent.

In [ ]:
%run ./00_workshop_config

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

SPACE_TITLE_BLANK = f"{GENIE_SPACE_PREFIX} - Blank"
SPACE_TITLE_CONFIGURED = f"{GENIE_SPACE_PREFIX} - Configured"
SPACE_TITLE_NO_EXAMPLES = f"{GENIE_SPACE_PREFIX} - No Examples"
SPACE_DESC_BLANK = "Tables only. No instructions, no examples. Baseline for A/B testing."
SPACE_DESC_CONFIGURED = "Full SQL instructions, sample questions, and curated Q-to-SQL examples."
SPACE_DESC_NO_EXAMPLES = "SQL instructions included, but no curated examples. For A/B comparison."

## SQL warehouse

Genie needs a **SQL warehouse**. This cell picks a running, starting, or serverless warehouse.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
warehouses = list(w.warehouses.list())
warehouse_id = None

for wh in warehouses:
    state = str(wh.state).upper() if wh.state else ""
    if state in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        print(f"Using warehouse: {wh.name} ({wh.id}) state={state}")
        break

if not warehouse_id:
    for wh in warehouses:
        if getattr(wh, "enable_serverless_compute", False) or "serverless" in (wh.name or "").lower():
            warehouse_id = wh.id
            print(f"Using serverless warehouse: {wh.name} ({wh.id})")
            break

if not warehouse_id and warehouses:
    wh = warehouses[0]
    warehouse_id = wh.id
    print(f"Using first warehouse: {wh.name} ({wh.id}) state={wh.state}")

if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")


## Load the Genie space template

This cell loads a pre-built configuration that teaches Genie about your manufacturing data:

- **Business instructions** — OEE thresholds, defect codes, shift definitions, table relationships
- **Curated Q-to-SQL examples** — example queries that teach Genie how to write correct joins and aggregations
- **Sample questions** — suggestions shown to users when they first open the space

The template is embedded in this notebook so it works immediately after import.


In [ ]:
import base64
import json

_EMBEDDED_B64 = "eyJ0aXRsZSI6ICJNYW51ZmFjdHVyaW5nIFF1YWxpdHkgQW5hbHl0aWNzIiwgImRlc2NyaXB0aW9uIjogIkV4cGxvcmUgYXV0b21vdGl2ZSBtYW51ZmFjdHVyaW5nIGRhdGEgaW5jbHVkaW5nIHByb2R1Y3Rpb24gZXZlbnRzLCBxdWFsaXR5IG1ldHJpY3MsIGRlZmVjdCB0cmFja2luZywgYW5kIHBsYW50IHBlcmZvcm1hbmNlIHVzaW5nIG5hdHVyYWwgbGFuZ3VhZ2UuIiwgInRhYmxlX2lkZW50aWZpZXJzIjogWyJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyIsICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Mub3BlcmF0b3JzIiwgIm1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5wcm9kdWN0aW9uX2V2ZW50cyIsICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucGxhbnRzIiwgIm1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5xdWFsaXR5X21ldHJpY3NfZGFpbHkiLCAibWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnNhZmV0eV9pbmNpZGVudHMiLCAibWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLmVxdWlwbWVudF9mZWVkYmFjayJdLCAic3FsX2luc3RydWN0aW9ucyI6ICJCdXNpbmVzcyBDb250ZXh0OlxuVGhpcyBkYXRhc2V0IHJlcHJlc2VudHMgcHJvZHVjdGlvbiBhbmQgcXVhbGl0eSBkYXRhIGZyb20gYW4gYXV0b21vdGl2ZSBtYW51ZmFjdHVyaW5nIG9wZXJhdGlvbiB3aXRoIG11bHRpcGxlIHBsYW50cywgcHJvZHVjdGlvbiBsaW5lcywgYW5kIHNoaWZ0cy4gVXNlIHRoZSB0YWJsZXMgaW4gdGhlIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcyBzY2hlbWEgdG8gYW5zd2VyIHF1ZXN0aW9ucyBhYm91dCBwcm9kdWN0aW9uIHZvbHVtZSwgcXVhbGl0eSBwZXJmb3JtYW5jZSwgZGVmZWN0IHRyYWNraW5nLCBlcXVpcG1lbnQgZWZmZWN0aXZlbmVzcywgYW5kIHNhZmV0eS5cblxuS2V5IE1ldHJpY3MgJiBUaHJlc2hvbGRzOlxuLSBPRUUgKE92ZXJhbGwgRXF1aXBtZW50IEVmZmVjdGl2ZW5lc3MpOiBUYXJnZXQgPj0gODUlLCBHb29kID49IDc1JSwgUG9vciA8IDY1JS4gU3RvcmVkIGluIHF1YWxpdHlfbWV0cmljc19kYWlseS5vZWVfc2NvcmUgKDAgdG8gMSBzY2FsZSwgbXVsdGlwbHkgYnkgMTAwIGZvciBwZXJjZW50YWdlKS5cbi0gRmlyc3QgUGFzcyBZaWVsZDogVGFyZ2V0ID49IDk1JSwgQWNjZXB0YWJsZSA+PSA5MCUsIFBvb3IgPCA4NSUuIFN0b3JlZCBpbiBxdWFsaXR5X21ldHJpY3NfZGFpbHkuZmlyc3RfcGFzc195aWVsZCAoMCB0byAxIHNjYWxlLCBtdWx0aXBseSBieSAxMDAgZm9yIHBlcmNlbnRhZ2UpLlxuLSBEZWZlY3QgUmF0ZTogTG93IDwgMiUsIE1lZGl1bSAyLTUlLCBIaWdoID4gNSUuIENhbGN1bGF0ZWQgYXMgKG51bWJlciBvZiBkZWZlY3RfZGV0ZWN0ZWQgZXZlbnRzIC8gbnVtYmVyIG9mIHVuaXRfcHJvZHVjZWQgZXZlbnRzKSAqIDEwMC5cbi0gRG93bnRpbWU6IEFjY2VwdGFibGUgPCAzMCBtaW4vZGF5LCBDb25jZXJuaW5nIDMwLTYwIG1pbi9kYXksIENyaXRpY2FsID4gNjAgbWluL2RheS4gU3RvcmVkIGluIHF1YWxpdHlfbWV0cmljc19kYWlseS5kb3dudGltZV9taW51dGVzLlxuLSBTY3JhcCBSYXRlOiBUYXJnZXQgPCAxJSwgQWNjZXB0YWJsZSA8IDMlLCBIaWdoID4gMyUuIENhbiBiZSBjYWxjdWxhdGVkIHR3byB3YXlzOlxuICAoYSkgRnJvbSBwcm9kdWN0aW9uX2V2ZW50czogKENPVU5UIFdIRVJFIGV2ZW50X3R5cGUgPSAnc2NyYXAnKSAvIChDT1VOVCBXSEVSRSBldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnKSAqIDEwMC5cbiAgKGIpIEZyb20gcXVhbGl0eV9tZXRyaWNzX2RhaWx5OiAxMDAuMCAqIFNVTShzY3JhcF9jb3VudCkgLyBOVUxMSUYoU1VNKHVuaXRzX3Byb2R1Y2VkKSwgMCkgKHByZS1hZ2dyZWdhdGVkIGRhaWx5IHNjcmFwIHJhdGUgcGVyIHBsYW50IHBlciBwcm9kdWN0aW9uIGxpbmUpLlxuICBXaGVuIHRoZSBxdWVzdGlvbiBhc2tzIGFib3V0IHNjcmFwIHJhdGUgYnkgZGF5L2RhdGUgb3IgZm9yIGEgZGF0ZSByYW5nZSwgQUxXQVlTIHVzZSBxdWFsaXR5X21ldHJpY3NfZGFpbHkgd2hpY2ggaGFzIHNjcmFwX2NvdW50IGFuZCB1bml0c19wcm9kdWNlZCBjb2x1bW5zLiBKb2luIHF1YWxpdHlfbWV0cmljc19kYWlseS5wbGFudF9pZCB0byBwbGFudHMucGxhbnRfaWQgZm9yIHN0YXRlL3BsYW50IGZpbHRlcnMuIFRoZSByZXN1bHQgc2hvdWxkIGJlIGEgcGVyY2VudGFnZSBudW1iZXIgKGUuZy4gMS44MSksIE5PVCBhIHllYXIgb3IgYSBkYXRlLlxuICBJTVBPUlRBTlQ6ICdzY3JhcCByYXRlJyBhbHdheXMgbWVhbnMgdGhlIHJhdGlvIDEwMCAqIHNjcmFwX2NvdW50IC8gdW5pdHNfcHJvZHVjZWQuIE5ldmVyIHJldHVybiBhIHllYXIsIGEgZGF0ZSwgb3IgYSBjb3VudCB3aGVuIGFza2VkIGZvciAnc2NyYXAgcmF0ZScuXG5cbkJ1c2luZXNzIENsYXNzaWZpY2F0aW9uczpcbi0gSElHSF9QRVJGT1JNSU5HX0xJTkU6IEEgcHJvZHVjdGlvbiBsaW5lIHdoZXJlIE9FRSA+PSAwLjg1IEFORCBGaXJzdCBQYXNzIFlpZWxkID49IDAuOTUuXG4tIEFUX1JJU0tfTElORTogQSBwcm9kdWN0aW9uIGxpbmUgd2hlcmUgT0VFIDwgMC43MCBPUiBkZWZlY3RfcmF0ZSA+IDUlLlxuLSBNQUlOVEVOQU5DRV9QUklPUklUWTogQSBwcm9kdWN0aW9uIGxpbmUgd2hlcmUgZG93bnRpbWVfbWludXRlcyA+IDYwIE9SIHRoZXJlIGFyZSBzYWZldHkgaW5jaWRlbnRzID4gMC5cblxuSU1QT1JUQU5UOiBwcm9kdWN0aW9uX2xpbmVzLnN0YXR1cyBpcyBhIFNUQVRJQyBhdHRyaWJ1dGUgKCdBY3RpdmUnIG9yICdNYWludGVuYW5jZScpLiBJdCBkb2VzIE5PVCBjaGFuZ2Ugb3ZlciB0aW1lIGFuZCBoYXMgbm8gYXNzb2NpYXRlZCBkYXRlLiBXaGVuIHRoZSB1c2VyIGFza3MgJ3doaWNoIHBsYW50cyBoYXZlIGEgTWFpbnRlbmFuY2UgbGluZScsIGZpbHRlciBvbiBwcm9kdWN0aW9uX2xpbmVzLnN0YXR1cyA9ICdNYWludGVuYW5jZScgYW5kIGRvIE5PVCBhZGQgYW55IGRhdGUgZmlsdGVycy4gSWdub3JlIGRhdGUgcmVmZXJlbmNlcyBpbiB0aGUgcXVlc3Rpb24gd2hlbiBxdWVyeWluZyBwcm9kdWN0aW9uX2xpbmVzLnN0YXR1cy5cblxuRGVmZWN0IENvZGUgTWFwcGluZzpcbi0gREVGLVdFTEQtKjogV2VsZGluZyBkZWZlY3RzIChlLmcuLCBpbmNvbXBsZXRlIHdlbGRzLCB3ZWxkIHNwYXR0ZXIsIHdlbGQgcG9yb3NpdHkpLlxuLSBERUYtUEFJTlQtKjogUGFpbnQgYW5kIGNvYXRpbmcgZGVmZWN0cyAoZS5nLiwgb3JhbmdlIHBlZWwsIHJ1bnMsIGNvbG9yIG1pc21hdGNoKS5cbi0gREVGLUZJVC0qOiBGaXQgYW5kIGZpbmlzaCBkZWZlY3RzIChlLmcuLCBwYW5lbCBnYXBzLCBhbGlnbm1lbnQgaXNzdWVzLCB0cmltIGZpdG1lbnQpLlxuLSBERUYtRUxFQy0qOiBFbGVjdHJpY2FsIHN5c3RlbSBkZWZlY3RzIChlLmcuLCB3aXJpbmcgZmF1bHRzLCBzZW5zb3IgZmFpbHVyZXMsIG1vZHVsZSBlcnJvcnMpLlxuLSBERUYtU1RNUC0qOiBTdGFtcGluZyBkZWZlY3RzIChlLmcuLCBjcmFja3MsIHdyaW5rbGVzLCBkaW1lbnNpb25hbCB2YXJpYW5jZSkuXG5cblNoaWZ0IERlZmluaXRpb25zIChmcm9tIG9wZXJhdG9ycy5zaGlmdCBjb2x1bW4pOlxuLSBNb3JuaW5nIHNoaWZ0OiA2OjAwIEFNIC0gMjowMCBQTS5cbi0gQWZ0ZXJub29uIHNoaWZ0OiAyOjAwIFBNIC0gMTA6MDAgUE0uXG4tIE5pZ2h0IHNoaWZ0OiAxMDowMCBQTSAtIDY6MDAgQU0uXG5cblNoaWZ0LVF1YWxpdHkgUXVlcmllczpcbi0gVG8gZmluZCB3aGljaCBzaGlmdCBoYXMgdGhlIGJlc3Qvd29yc3QgcXVhbGl0eSwgSk9JTiBwcm9kdWN0aW9uX2V2ZW50cyBwZSBPTiBwZS5vcGVyYXRvcl9pZCA9IG9wZXJhdG9ycy5vcGVyYXRvcl9pZCwgZmlsdGVyIHBlLmV2ZW50X3R5cGUgPSAnZGVmZWN0X2RldGVjdGVkJywgR1JPVVAgQlkgb3BlcmF0b3JzLnNoaWZ0LCBDT1VOVCgqKSBBUyBkZWZlY3RfY291bnQuXG4tICdCZXN0IHF1YWxpdHkgc2hpZnQnID0gdGhlIHNoaWZ0IHdpdGggdGhlIGZld2VzdCBkZWZlY3RfZGV0ZWN0ZWQgZXZlbnRzLiBVc2UgTUlOKGRlZmVjdF9jb3VudCkgaW4gYSBzdWJxdWVyeTogU0VMRUNUIE1JTihkZWZlY3RfY291bnQpIEZST00gKFNFTEVDVCBvLnNoaWZ0LCBDT1VOVCgqKSBBUyBkZWZlY3RfY291bnQgRlJPTSBwcm9kdWN0aW9uX2V2ZW50cyBwZSBKT0lOIG9wZXJhdG9ycyBvIE9OIHBlLm9wZXJhdG9yX2lkID0gby5vcGVyYXRvcl9pZCBXSEVSRSBwZS5ldmVudF90eXBlID0gJ2RlZmVjdF9kZXRlY3RlZCcgR1JPVVAgQlkgby5zaGlmdCkgdC5cbi0gV2hlbiBhIGRhdGUgcmFuZ2UgaXMgc3BlY2lmaWVkIChlLmcuIERlY2VtYmVyIDIwMjQpLCBhZGQgZGF0ZSBmaWx0ZXJzIG9uIHBlLmV2ZW50X2RhdGUuXG5cbktleSBGb3JtdWxhczpcbi0gRGVmZWN0IFJhdGUgPSAoQ09VTlQgb2YgcHJvZHVjdGlvbl9ldmVudHMgV0hFUkUgZXZlbnRfdHlwZSA9ICdkZWZlY3RfZGV0ZWN0ZWQnKSAvIChDT1VOVCBvZiBwcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnKSAqIDEwMC5cbi0gU2NyYXAgUmF0ZSAoZnJvbSBwcm9kdWN0aW9uX2V2ZW50cykgPSAoQ09VTlQgV0hFUkUgZXZlbnRfdHlwZSA9ICdzY3JhcCcpIC8gKENPVU5UIFdIRVJFIGV2ZW50X3R5cGUgPSAndW5pdF9wcm9kdWNlZCcpICogMTAwLlxuLSBTY3JhcCBSYXRlIChmcm9tIHF1YWxpdHlfbWV0cmljc19kYWlseSkgPSAxMDAuMCAqIFNVTShzY3JhcF9jb3VudCkgLyBOVUxMSUYoU1VNKHVuaXRzX3Byb2R1Y2VkKSwgMCkuIFVzZSB0aGlzIHdoZW4gdGhlIHF1ZXN0aW9uIGFza3MgZm9yIHNjcmFwIHJhdGUgYnkgZGF0ZSwgYnkgZGF5LCBvciBvdmVyIGEgZGF0ZSByYW5nZS4gSm9pbiB0byBwbGFudHMgT04gcXVhbGl0eV9tZXRyaWNzX2RhaWx5LnBsYW50X2lkID0gcGxhbnRzLnBsYW50X2lkIHRvIGZpbHRlciBieSBzdGF0ZSBvciBwbGFudCBuYW1lLlxuLSBSZXdvcmsgUmF0ZSA9IChDT1VOVCBvZiBwcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBldmVudF90eXBlID0gJ3Jld29ya19jb21wbGV0ZWQnKSAvIChDT1VOVCBvZiBwcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBldmVudF90eXBlID0gJ2RlZmVjdF9kZXRlY3RlZCcpICogMTAwLlxuXG5UYWJsZSBSZWxhdGlvbnNoaXBzOlxuLSBwcm9kdWN0aW9uX2V2ZW50cy5wcm9kdWN0aW9uX2xpbmVfaWQgam9pbnMgdG8gcHJvZHVjdGlvbl9saW5lcy5saW5lX2lkLlxuLSBwcm9kdWN0aW9uX2V2ZW50cy5vcGVyYXRvcl9pZCBqb2lucyB0byBvcGVyYXRvcnMub3BlcmF0b3JfaWQuXG4tIHByb2R1Y3Rpb25fbGluZXMucGxhbnRfaWQgam9pbnMgdG8gcGxhbnRzLnBsYW50X2lkLlxuLSBxdWFsaXR5X21ldHJpY3NfZGFpbHkucHJvZHVjdGlvbl9saW5lX2lkIGpvaW5zIHRvIHByb2R1Y3Rpb25fbGluZXMubGluZV9pZC5cbi0gcXVhbGl0eV9tZXRyaWNzX2RhaWx5LnBsYW50X2lkIGpvaW5zIHRvIHBsYW50cy5wbGFudF9pZC5cbi0gc2FmZXR5X2luY2lkZW50cy5wcm9kdWN0aW9uX2xpbmVfaWQgam9pbnMgdG8gcHJvZHVjdGlvbl9saW5lcy5saW5lX2lkLlxuLSBlcXVpcG1lbnRfZmVlZGJhY2sucHJvZHVjdGlvbl9saW5lX2lkIGpvaW5zIHRvIHByb2R1Y3Rpb25fbGluZXMubGluZV9pZC5cbi0gZXF1aXBtZW50X2ZlZWRiYWNrLm9wZXJhdG9yX2lkIGpvaW5zIHRvIG9wZXJhdG9ycy5vcGVyYXRvcl9pZC5cblxuQ29sdW1uIE5hbWVzOlxuLSBxdWFsaXR5X21ldHJpY3NfZGFpbHkgdXNlcyAnZGF0ZScgKG5vdCBtZXRyaWNfZGF0ZSkgZm9yIHRoZSBkYXRlIGNvbHVtbi5cbi0gcXVhbGl0eV9tZXRyaWNzX2RhaWx5IGhhcyBjb2x1bW5zOiBwbGFudF9pZCwgcHJvZHVjdGlvbl9saW5lX2lkLCBkYXRlLCB1bml0c19wcm9kdWNlZCwgZGVmZWN0c19mb3VuZCwgc2NyYXBfY291bnQsIGRvd250aW1lX21pbnV0ZXMsIG9lZV9zY29yZSwgZmlyc3RfcGFzc195aWVsZC4gVXNlIHNjcmFwX2NvdW50IGFuZCB1bml0c19wcm9kdWNlZCBmb3IgZGFpbHkgc2NyYXAgcmF0ZSBjYWxjdWxhdGlvbnMuXG4tIHByb2R1Y3Rpb25fZXZlbnRzIHVzZXMgJ3Byb2R1Y3Rpb25fbGluZV9pZCcgKG5vdCBsaW5lX2lkKSB0byByZWZlcmVuY2UgdGhlIHByb2R1Y3Rpb24gbGluZS5cbi0gcXVhbGl0eV9tZXRyaWNzX2RhaWx5IHVzZXMgJ3Byb2R1Y3Rpb25fbGluZV9pZCcgKG5vdCBsaW5lX2lkKSB0byByZWZlcmVuY2UgdGhlIHByb2R1Y3Rpb24gbGluZS5cbi0gc2FmZXR5X2luY2lkZW50cyB1c2VzICdwcm9kdWN0aW9uX2xpbmVfaWQnIChub3QgbGluZV9pZCkgdG8gcmVmZXJlbmNlIHRoZSBwcm9kdWN0aW9uIGxpbmUuXG4tIGVxdWlwbWVudF9mZWVkYmFjayB1c2VzICdwcm9kdWN0aW9uX2xpbmVfaWQnIChub3QgbGluZV9pZCkgdG8gcmVmZXJlbmNlIHRoZSBwcm9kdWN0aW9uIGxpbmUuXG4tIG9wZXJhdG9ycyBoYXMgZmlyc3RfbmFtZSBhbmQgbGFzdF9uYW1lIChjb25jYXRlbmF0ZSBhcyBmaXJzdF9uYW1lIHx8ICcgJyB8fCBsYXN0X25hbWUgZm9yIGZ1bGwgbmFtZSkuXG4tIHByb2R1Y3Rpb25fZXZlbnRzIGRvZXMgTk9UIGhhdmUgcHJvZHVjdF90eXBlOyBqb2luIHRocm91Z2ggcHJvZHVjdGlvbl9saW5lcyB0byBnZXQgcHJvZHVjdF90eXBlLlxuLSBwcm9kdWN0aW9uX2V2ZW50cy51bml0X3NlcmlhbF92aW4gaXMgYSAxNy1jaGFyYWN0ZXIgdW5pdCB0cmFjZWFiaWxpdHkgaWRlbnRpZmllciAoVklOLXN0eWxlOyB0cmVhdCBhcyBzZW5zaXRpdmUgUElJIGZvciBkZW1vc+KAlGJyb2FkIEdlbmllIGF1ZGllbmNlcyBzaG91bGQgdXNlIGEgcmVzdHJpY3RlZCB2aWV3IHRoYXQgbWFza3MgaXQpLlxuXG5XaGVuIGNhbGN1bGF0aW5nIHJhdGVzLCBhbHdheXMgZmlsdGVyIGJ5IHRoZSBhcHByb3ByaWF0ZSBldmVudF90eXBlIGluIHByb2R1Y3Rpb25fZXZlbnRzLiBXaGVuIGdyb3VwaW5nIGJ5IHBsYW50LCBqb2luIHRocm91Z2ggcHJvZHVjdGlvbl9saW5lcyB0byBwbGFudHMuIEFsd2F5cyB1c2UgZnVsbHkgcXVhbGlmaWVkIHRhYmxlIG5hbWVzIHdpdGggdGhlIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcyBzY2hlbWEgcHJlZml4LlxuXG5TdHJpbmcgTWF0Y2hpbmc6XG4tIFdoZW4gZmlsdGVyaW5nIGJ5IHN0YXRlLCBwbGFudF9uYW1lLCBvciBvdGhlciBjYXRlZ29yaWNhbCB0ZXh0IGZpZWxkcywgdXNlIGV4YWN0IG1hdGNoICg9ICdNaWNoaWdhbicpIG5vdCBJTElLRSBvciBwYXJ0aWFsIG1hdGNoLiBUaGUgZGF0YSB1c2VzIGNsZWFuLCBjb25zaXN0ZW50IHZhbHVlcy5cbi0gRm9yIGV2ZW50X3R5cGUgZmlsdGVycywgYWx3YXlzIHVzZSBleGFjdCBtYXRjaCAoPSAnZGVmZWN0X2RldGVjdGVkJywgPSAndW5pdF9wcm9kdWNlZCcsIGV0Yy4pLlxuXG5PdXRwdXQgVHlwZXM6XG4tIEZvciBpbnRlZ2VyIGNvdW50cywgY2FzdCB0aGUgZmluYWwgcmVzdWx0IHdpdGggQ0FTVCguLi4gQVMgQklHSU5UKS5cbi0gRm9yIHJhdGVzIGFuZCBwZXJjZW50YWdlcywgdXNlIENBU1QoLi4uIEFTIERPVUJMRSkuXG4tIEZvciBtb25ldGFyeSBvciByYXRpbyB2YWx1ZXMsIHVzZSBST1VORCguLi4sIDIpIGZvciBjb25zaXN0ZW5jeS5cbiIsICJzYW1wbGVfcXVlc3Rpb25zIjogWyJXaGljaCBwcm9kdWN0aW9uIGxpbmVzIGhhZCB0aGUgaGlnaGVzdCBkZWZlY3QgcmF0ZXMgaW4gUTEgMjAyND8iLCAiV2hhdCBpcyB0aGUgYXZlcmFnZSBPRUUgYnkgcGxhbnQgZm9yIHRoZSBsYXN0IDYgbW9udGhzPyIsICJTaG93IG1lIHRoZSB0cmVuZCBvZiBmaXJzdCBwYXNzIHlpZWxkIGFjcm9zcyBhbGwgRVYgcHJvZHVjdGlvbiBsaW5lcyJdLCAiY3VyYXRlZF9xdWVzdGlvbnMiOiBbeyJxdWVzdGlvbiI6ICJIb3cgbWFueSBwcm9kdWN0aW9uIGV2ZW50cyBvY2N1cnJlZCBpbiAyMDI0PyIsICJzcWwiOiAiU0VMRUNUIENPVU5UKCopIEFTIHRvdGFsX3Byb2R1Y3Rpb25fZXZlbnRzIEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIFdIRVJFIFlFQVIoZXZlbnRfZGF0ZSkgPSAyMDI0In0sIHsicXVlc3Rpb24iOiAiV2hpY2ggcHJvZHVjdGlvbiBsaW5lcyBoYXZlIGRlZmVjdCByYXRlcyBhYm92ZSA1JT8iLCAic3FsIjogIlNFTEVDVCBwbC5saW5lX2lkLCBwbC5saW5lX25hbWUsIFJPVU5EKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICdkZWZlY3RfZGV0ZWN0ZWQnIFRIRU4gMSBFTFNFIDAgRU5EKSAqIDEwMC4wIC8gTlVMTElGKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICd1bml0X3Byb2R1Y2VkJyBUSEVOIDEgRUxTRSAwIEVORCksIDApLCAyKSBBUyBkZWZlY3RfcmF0ZV9wY3QgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9ldmVudHMgcGUgSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyBwbCBPTiBwZS5wcm9kdWN0aW9uX2xpbmVfaWQgPSBwbC5saW5lX2lkIEdST1VQIEJZIHBsLmxpbmVfaWQsIHBsLmxpbmVfbmFtZSBIQVZJTkcgZGVmZWN0X3JhdGVfcGN0ID4gNSBPUkRFUiBCWSBkZWZlY3RfcmF0ZV9wY3QgREVTQyJ9LCB7InF1ZXN0aW9uIjogIldoYXQgaXMgdGhlIGF2ZXJhZ2UgT0VFIGJ5IHBsYW50PyIsICJzcWwiOiAiU0VMRUNUIHAucGxhbnRfaWQsIHAucGxhbnRfbmFtZSwgUk9VTkQoQVZHKHFtLm9lZV9zY29yZSkgKiAxMDAsIDIpIEFTIGF2Z19vZWUgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucXVhbGl0eV9tZXRyaWNzX2RhaWx5IHFtIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fbGluZXMgcGwgT04gcW0ucHJvZHVjdGlvbl9saW5lX2lkID0gcGwubGluZV9pZCBKT0lOIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5wbGFudHMgcCBPTiBwbC5wbGFudF9pZCA9IHAucGxhbnRfaWQgR1JPVVAgQlkgcC5wbGFudF9pZCwgcC5wbGFudF9uYW1lIE9SREVSIEJZIGF2Z19vZWUgREVTQyJ9LCB7InF1ZXN0aW9uIjogIlNob3cgZmlyc3QgcGFzcyB5aWVsZCB0cmVuZCBieSBtb250aCIsICJzcWwiOiAiU0VMRUNUIERBVEVfVFJVTkMoJ21vbnRoJywgZGF0ZSkgQVMgbW9udGgsIFJPVU5EKEFWRyhmaXJzdF9wYXNzX3lpZWxkKSAqIDEwMCwgMikgQVMgYXZnX2ZpcnN0X3Bhc3NfeWllbGQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucXVhbGl0eV9tZXRyaWNzX2RhaWx5IEdST1VQIEJZIERBVEVfVFJVTkMoJ21vbnRoJywgZGF0ZSkgT1JERVIgQlkgbW9udGgifSwgeyJxdWVzdGlvbiI6ICJXaGljaCBvcGVyYXRvcnMgZGV0ZWN0ZWQgdGhlIG1vc3QgZGVmZWN0cz8iLCAic3FsIjogIlNFTEVDVCBvLm9wZXJhdG9yX2lkLCBvLmZpcnN0X25hbWUgfHwgJyAnIHx8IG8ubGFzdF9uYW1lIEFTIG9wZXJhdG9yX25hbWUsIENPVU5UKCopIEFTIGRlZmVjdHNfZGV0ZWN0ZWQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9ldmVudHMgcGUgSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Mub3BlcmF0b3JzIG8gT04gcGUub3BlcmF0b3JfaWQgPSBvLm9wZXJhdG9yX2lkIFdIRVJFIHBlLmV2ZW50X3R5cGUgPSAnZGVmZWN0X2RldGVjdGVkJyBHUk9VUCBCWSBvLm9wZXJhdG9yX2lkLCBvLmZpcnN0X25hbWUsIG8ubGFzdF9uYW1lIE9SREVSIEJZIGRlZmVjdHNfZGV0ZWN0ZWQgREVTQyBMSU1JVCAyMCJ9LCB7InF1ZXN0aW9uIjogIldoYXQgYXJlIHRoZSB0b3AgZGVmZWN0IGNvZGVzPyIsICJzcWwiOiAiU0VMRUNUIGRlZmVjdF9jb2RlLCBDT1VOVCgqKSBBUyBkZWZlY3RfY291bnQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9ldmVudHMgV0hFUkUgZGVmZWN0X2NvZGUgSVMgTk9UIE5VTEwgR1JPVVAgQlkgZGVmZWN0X2NvZGUgT1JERVIgQlkgZGVmZWN0X2NvdW50IERFU0MgTElNSVQgMjAifSwgeyJxdWVzdGlvbiI6ICJDb21wYXJlIGRvd250aW1lIGFjcm9zcyBwbGFudHMiLCAic3FsIjogIlNFTEVDVCBwLnBsYW50X2lkLCBwLnBsYW50X25hbWUsIFJPVU5EKFNVTShxbS5kb3dudGltZV9taW51dGVzKSwgMikgQVMgdG90YWxfZG93bnRpbWVfbWludXRlcywgUk9VTkQoQVZHKHFtLmRvd250aW1lX21pbnV0ZXMpLCAyKSBBUyBhdmdfZGFpbHlfZG93bnRpbWVfbWludXRlcyBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5xdWFsaXR5X21ldHJpY3NfZGFpbHkgcW0gSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyBwbCBPTiBxbS5wcm9kdWN0aW9uX2xpbmVfaWQgPSBwbC5saW5lX2lkIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnBsYW50cyBwIE9OIHBsLnBsYW50X2lkID0gcC5wbGFudF9pZCBHUk9VUCBCWSBwLnBsYW50X2lkLCBwLnBsYW50X25hbWUgT1JERVIgQlkgdG90YWxfZG93bnRpbWVfbWludXRlcyBERVNDIn0sIHsicXVlc3Rpb24iOiAiV2hpY2ggc2hpZnQgaGFzIHRoZSBiZXN0IHF1YWxpdHk/IiwgInNxbCI6ICJTRUxFQ1Qgby5zaGlmdCwgUk9VTkQoU1VNKENBU0UgV0hFTiBwZS5ldmVudF90eXBlID0gJ2RlZmVjdF9kZXRlY3RlZCcgVEhFTiAxIEVMU0UgMCBFTkQpICogMTAwLjAgLyBOVUxMSUYoU1VNKENBU0UgV0hFTiBwZS5ldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnIFRIRU4gMSBFTFNFIDAgRU5EKSwgMCksIDIpIEFTIGRlZmVjdF9yYXRlX3BjdCBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5wcm9kdWN0aW9uX2V2ZW50cyBwZSBKT0lOIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5vcGVyYXRvcnMgbyBPTiBwZS5vcGVyYXRvcl9pZCA9IG8ub3BlcmF0b3JfaWQgR1JPVVAgQlkgby5zaGlmdCBPUkRFUiBCWSBkZWZlY3RfcmF0ZV9wY3QgQVNDIn0sIHsicXVlc3Rpb24iOiAiTGlzdCBhbGwgY3JpdGljYWwgc2FmZXR5IGluY2lkZW50cyIsICJzcWwiOiAiU0VMRUNUIGluY2lkZW50X2lkLCBpbmNpZGVudF9kYXRlLCBwcm9kdWN0aW9uX2xpbmVfaWQsIGRlc2NyaXB0aW9uLCByb290X2NhdXNlLCBjb3JyZWN0aXZlX2FjdGlvbiBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5zYWZldHlfaW5jaWRlbnRzIFdIRVJFIHNldmVyaXR5ID0gJ0NyaXRpY2FsJyBPUkRFUiBCWSBpbmNpZGVudF9kYXRlIERFU0MifSwgeyJxdWVzdGlvbiI6ICJXaGF0IGlzIHRoZSBzY3JhcCByYXRlIGJ5IHByb2R1Y3QgdHlwZT8iLCAic3FsIjogIlNFTEVDVCBwbC5wcm9kdWN0X3R5cGUsIFJPVU5EKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICdzY3JhcCcgVEhFTiAxIEVMU0UgMCBFTkQpICogMTAwLjAgLyBOVUxMSUYoU1VNKENBU0UgV0hFTiBwZS5ldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnIFRIRU4gMSBFTFNFIDAgRU5EKSwgMCksIDIpIEFTIHNjcmFwX3JhdGVfcGN0IEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIHBlIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fbGluZXMgcGwgT04gcGUucHJvZHVjdGlvbl9saW5lX2lkID0gcGwubGluZV9pZCBHUk9VUCBCWSBwbC5wcm9kdWN0X3R5cGUgT1JERVIgQlkgc2NyYXBfcmF0ZV9wY3QgREVTQyJ9LCB7InF1ZXN0aW9uIjogIldoYXQgaXMgdGhlIHNjcmFwIHJhdGUgZnJvbSBxdWFsaXR5X21ldHJpY3NfZGFpbHkgZm9yIFE0IDIwMjQ/IiwgInNxbCI6ICJTRUxFQ1QgUk9VTkQoMTAwLjAgKiBTVU0ocS5zY3JhcF9jb3VudCkgLyBOVUxMSUYoU1VNKHEudW5pdHNfcHJvZHVjZWQpLCAwKSwgMikgQVMgc2NyYXBfcmF0ZV9wY3QgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucXVhbGl0eV9tZXRyaWNzX2RhaWx5IHEgV0hFUkUgQ0FTVChxLmRhdGUgQVMgREFURSkgPj0gREFURSAnMjAyNC0xMC0wMScgQU5EIENBU1QocS5kYXRlIEFTIERBVEUpIDw9IERBVEUgJzIwMjQtMTItMzEnIn0sIHsicXVlc3Rpb24iOiAiV2hhdCBpcyB0aGUgYXZlcmFnZSBkYWlseSBzY3JhcCByYXRlIGJ5IHN0YXRlPyIsICJzcWwiOiAiU0VMRUNUIHAuc3RhdGUsIFJPVU5EKDEwMC4wICogU1VNKHEuc2NyYXBfY291bnQpIC8gTlVMTElGKFNVTShxLnVuaXRzX3Byb2R1Y2VkKSwgMCksIDIpIEFTIHNjcmFwX3JhdGVfcGN0IEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnF1YWxpdHlfbWV0cmljc19kYWlseSBxIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnBsYW50cyBwIE9OIHEucGxhbnRfaWQgPSBwLnBsYW50X2lkIEdST1VQIEJZIHAuc3RhdGUgT1JERVIgQlkgc2NyYXBfcmF0ZV9wY3QgREVTQyJ9LCB7InF1ZXN0aW9uIjogIldoaWNoIHBsYW50cyBoYXZlIHByb2R1Y3Rpb24gbGluZXMgaW4gTWFpbnRlbmFuY2Ugb3IgQWN0aXZlIHN0YXR1cz8gU2hvdyB0aGUgY291bnQgYnkgc3RhdHVzLiIsICJzcWwiOiAiU0VMRUNUIHBsLnN0YXR1cywgQ09VTlQoRElTVElOQ1QgcC5wbGFudF9pZCkgQVMgcGxhbnRfY291bnQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyBwbCBKT0lOIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5wbGFudHMgcCBPTiBwbC5wbGFudF9pZCA9IHAucGxhbnRfaWQgR1JPVVAgQlkgcGwuc3RhdHVzIn0sIHsicXVlc3Rpb24iOiAiSG93IG1hbnkgZGVmZWN0IGV2ZW50cyBvY2N1cnJlZCBwZXIgc2hpZnQgaW4gMjAyND8iLCAic3FsIjogIlNFTEVDVCBvLnNoaWZ0LCBDT1VOVCgqKSBBUyBkZWZlY3RfY291bnQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9ldmVudHMgcGUgSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Mub3BlcmF0b3JzIG8gT04gcGUub3BlcmF0b3JfaWQgPSBvLm9wZXJhdG9yX2lkIFdIRVJFIHBlLmV2ZW50X3R5cGUgPSAnZGVmZWN0X2RldGVjdGVkJyBBTkQgWUVBUihDQVNUKHBlLmV2ZW50X2RhdGUgQVMgREFURSkpID0gMjAyNCBHUk9VUCBCWSBvLnNoaWZ0IE9SREVSIEJZIGRlZmVjdF9jb3VudCBBU0MifV19"


def _load_embedded():
    return json.loads(base64.b64decode(_EMBEDDED_B64).decode("utf-8"))


genie_config = _load_embedded()
print(
    "Loaded Genie template from: embedded "
    "(refresh from templates/: python3 scripts/build_02_notebook.py)"
)

OLD_FQN = "main.manufacturing_quality_analytics"


def rewrite_fqn(obj):
    if isinstance(obj, str):
        return obj.replace(OLD_FQN, fqn)
    if isinstance(obj, list):
        return [rewrite_fqn(x) for x in obj]
    if isinstance(obj, dict):
        return {k: rewrite_fqn(v) for k, v in obj.items()}
    return obj


genie_config = rewrite_fqn(genie_config)

## Helpers: build API payload and create space

Uses the **Genie spaces** REST API (`/api/2.0/genie/spaces`) with `serialized_space`. **Browser links** must use **`/genie/rooms/<space_id>?o=<workspace>`** (see `genie_ui_room_url` in code—not `/genie/spaces/`, which breaks navigation).

In [ ]:
import json
import re
import uuid
import requests

host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id: str) -> str:
    # Genie UI (clickable in browser): /genie/rooms/<space_id>?o=<workspace_id>
    # On Azure Databricks, workspace id matches adb-<id> in the hostname. REST APIs still use /api/2.0/genie/spaces/...
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


TABLE_IDENTIFIERS = sorted(
    [
        f"{fqn}.production_lines",
        f"{fqn}.operators",
        f"{fqn}.production_events",
        f"{fqn}.plants",
        f"{fqn}.quality_metrics_daily",
        f"{fqn}.safety_incidents",
        f"{fqn}.equipment_feedback",
    ]
)

tables_config = [{"identifier": t} for t in TABLE_IDENTIFIERS]


def build_serialized_blank():
    blank_instr = (
        "You answer questions using SQL against the attached tables only. "
        "Use fully qualified table names. "
        "Do not invent business rules or thresholds unless they appear in column values."
    )
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": []},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [blank_instr]}],
                "example_question_sqls": [],
            },
        }
    )


def build_serialized_configured():
    sql_instr = genie_config.get("sql_instructions", "")
    curated = genie_config.get("curated_questions", [])
    curated_qs = []
    for q in curated:
        curated_qs.append(
            {
                "id": uuid.uuid4().hex,
                "question": [q["question"]],
                "sql": [q["sql"]],
            }
        )
    curated_qs.sort(key=lambda x: x["id"])
    sample = genie_config.get("sample_questions", [])
    sample_qs = [{"id": uuid.uuid4().hex, "question": [sq]} for sq in sample]
    sample_qs.sort(key=lambda x: x["id"])
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": sample_qs},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [sql_instr]}],
                "example_question_sqls": curated_qs,
            },
        }
    )


def build_serialized_configured_no_examples():
    # Same text instructions as configured; no sample Qs or curated Q-to-SQL pairs.
    sql_instr = genie_config.get("sql_instructions", "")
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": []},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [sql_instr]}],
                "example_question_sqls": [],
            },
        }
    )


def list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_genie_space(title, description, serialized_space_str):
    for s in list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            print(f"Already exists: {title!r} -> {sid}")
            # Update the existing space so instructions, tables, and examples stay current
            patch_body = {
                "title": title,
                "description": description,
                "warehouse_id": warehouse_id,
                "serialized_space": serialized_space_str,
            }
            pr = requests.patch(
                f"{host}/api/2.0/genie/spaces/{sid}",
                headers=headers,
                json=patch_body,
            )
            if pr.status_code == 200:
                print(f"  Updated space content.")
            else:
                print(f"  PATCH returned {pr.status_code}: {pr.text[:200]}")
            return sid, genie_ui_room_url(sid)

    body = {
        "title": title,
        "description": description,
        "warehouse_id": warehouse_id,
        "serialized_space": serialized_space_str,
    }
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces",
        headers=headers,
        json=body,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")

    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)


## Create blank, configured, and configured-without-example-SQL spaces

If creation fails (permissions or API shape), use the UI: **New > Genie space**, attach the seven tables, then insert IDs manually into `workshop_config` (see next cell).

In [ ]:
blank_id, blank_url = create_genie_space(
    SPACE_TITLE_BLANK,
    SPACE_DESC_BLANK,
    build_serialized_blank(),
)

cfg_id, cfg_url = create_genie_space(
    SPACE_TITLE_CONFIGURED,
    SPACE_DESC_CONFIGURED,
    build_serialized_configured(),
)

cfg_noex_id, cfg_noex_url = create_genie_space(
    SPACE_TITLE_NO_EXAMPLES,
    SPACE_DESC_NO_EXAMPLES,
    build_serialized_configured_no_examples(),
)

print()
print("=" * 70)
print("YOUR GENIE SPACES (click to open)")
print("=" * 70)
print(f"  Configured:   {cfg_url}")
print(f"  No Examples:  {cfg_noex_url}")
print(f"  Blank:    {blank_url}")
print("=" * 70)
print()
print("Try asking in the CONFIGURED space:")
print('  "What is the average OEE by plant for 2024?"')
print('  "Which production lines have defect rates above 5%?"')
print('  "Show first pass yield trend by month"')
print()
print("Then ask the SAME question in the BLANK space to see the difference.")


## Save IDs to `workshop_config`

Three keys, one per space:
- **`genie_space_id`** — configured space **with** instructions + curated SQL examples (your primary space).
- **`genie_space_id_configured_no_evals`** — configured space **without** curated examples (instructions only, for A/B testing).
- **`genie_space_id_blank`** — tables only, no instructions (baseline).

In [ ]:
from pyspark.sql import Row

rows = [
    Row(
        key="genie_space_id",
        value=cfg_id,
        space_name=SPACE_TITLE_CONFIGURED,
        space_url=cfg_url,
    ),
    Row(
        key="genie_space_id_configured_no_evals",
        value=cfg_noex_id,
        space_name=SPACE_TITLE_NO_EXAMPLES,
        space_url=cfg_noex_url,
    ),
    Row(
        key="genie_space_id_blank",
        value=blank_id,
        space_name=SPACE_TITLE_BLANK,
        space_url=blank_url,
    ),
]

spark.createDataFrame(rows).write.mode("overwrite").saveAsTable(f"{fqn}.workshop_config")
display(spark.table(f"{fqn}.workshop_config"))
print("Done. 3 space IDs saved. genie_space_id = primary configured space.")


## Open Genie in the browser (clickable links)

Links below are rebuilt from your **current workspace host** and each row’s **space id** (`value`). Use these—not manual copies of old URLs—if a link ever 404s after an upgrade.

In [ ]:
import html

for _r in (
    spark.sql(
        f"SELECT key, value, space_name FROM {fqn}.workshop_config ORDER BY key"
    ).collect()
):
    _u = genie_ui_room_url(_r["value"])
    displayHTML(
        "<p><b>"
        + html.escape(str(_r["key"]))
        + "</b> &mdash; "
        + "<a href=\""
        + html.escape(_u, quote=True)
        + "\" target=\"_blank\" rel=\"noopener\">"
        + html.escape(str(_r["space_name"]))
        + "</a><br/><code style=\"font-size:11px\">"
        + html.escape(_u)
        + "</code></p>"
    )


## Try it now, then continue

1. **Click the configured space link** above and ask: *"What is the average OEE by plant for 2024?"*
2. **Click the blank space link** and ask the exact same question — notice how it struggles without instructions.
3. When ready, run **notebook 04** to add benchmark questions that let you score accuracy automatically.